# Day 2 — Python: String Methods & Manipulation
### Target: Data Engineer Interviews

> **Roadmap Day:** 2 · **Date:** Tuesday, June 16, 2026  
> **Prerequisite:** Run `00_prerequisites.ipynb` first.

Run each cell in order. Try predicting the output before running each cell.

---
## 1. Strings Are Immutable

Every string method returns a **new** string — the original is never changed.  
You must assign the result back or chain methods.

In [1]:
s = "  Hello, World!  "
result = s.strip()

print(f"original : {repr(s)}")
print(f"stripped : {repr(result)}")
print(f"original unchanged: {repr(s)}")

original : '  Hello, World!  '
stripped : 'Hello, World!'
original unchanged: '  Hello, World!  '


In [2]:
# Chain methods — each returns a new string for the next call
email = "   ALICE@Gmail.COM   "
clean = email.strip().lower()
print(clean)   # alice@gmail.com

alice@gmail.com


---
## 2. Case Conversion — upper, lower, title, capitalize

| Method | Effect |
|--------|--------|
| `upper()` | ALL CAPS |
| `lower()` | all lowercase |
| `title()` | First Letter Of Each Word |
| `capitalize()` | First letter only |
| `swapcase()` | Flip each character |

**DE context:** Standardise department names, email addresses, city names before DB insert or comparison.

In [5]:
name = "alice johnson"

print(name.upper())       # ALICE JOHNSON
print(name.lower())       # alice johnson
print(name.title())       # Alice Johnson
print(name.capitalize())  # Alice johnson  ← only first letter
print("ALICE johNSON".swapcase())  # alice johnson

ALICE JOHNSON
alice johnson
Alice Johnson
Alice johnson
alice JOHnson


In [6]:
# DE use case: case-insensitive comparison
dept_from_db  = "ENGINEERING"
dept_from_api = "engineering"

# Wrong — these look different
print(dept_from_db.lower() == dept_from_api.lower())   # True  ← correct way
print(dept_from_db == dept_from_api)                   # False ← wrong

True
False


In [7]:
# Real dataset: messy emails
customers = [
    {"customer_id": 1, "email": "ALICE@Gmail.COM   "},
    {"customer_id": 2, "email": "bob@yahoo.com"},
    {"customer_id": 3, "email": " carol@hotmail.com"},
    {"customer_id": 4, "email": "david@gmail.com"},
    {"customer_id": 5, "email": "EVE@GMAIL.COM"},
]

for c in customers:
    clean = c["email"].strip().lower()
    print(f"customer_id={c['customer_id']} | raw={c['email']:<20} | clean={clean}")

customer_id=1 | raw=ALICE@Gmail.COM      | clean=alice@gmail.com
customer_id=2 | raw=bob@yahoo.com        | clean=bob@yahoo.com
customer_id=3 | raw= carol@hotmail.com   | clean=carol@hotmail.com
customer_id=4 | raw=david@gmail.com      | clean=david@gmail.com
customer_id=5 | raw=EVE@GMAIL.COM        | clean=eve@gmail.com


---
## 3. strip / lstrip / rstrip — Whitespace Removal

`strip()` removes **leading and trailing** whitespace (spaces, tabs, newlines).  
You can also strip specific characters by passing them as an argument.

In [8]:
raw = "   data engineer   "

print(repr(raw.strip()))   # 'data engineer'
print(repr(raw.lstrip()))  # 'data engineer   '
print(repr(raw.rstrip()))  # '   data engineer'

'data engineer'
'data engineer   '
'   data engineer'


In [10]:
# Strip specific characters (not substrings!)
tag = "###IMPORTANT###"
print(tag.strip("#"))       # IMPORTANT

phone = "+91-98765-43210"
print(phone.strip("+"))     # 91-98765-43210  ← only strips leading/trailing '+'

IMPORTANT
91-98765-43210


In [11]:
# Why strip matters — equality check without strip fails
name_raw = "  Alice Johnson  "

print(f"Before strip: name={repr(name_raw)}, len={len(name_raw)}")
print(f"After strip:  name={repr(name_raw.strip())}, len={len(name_raw.strip())}")

Before strip: name='  Alice Johnson  ', len=17
After strip:  name='Alice Johnson', len=13


---
## 4. split() — Break String Into a List

`str.split(sep=None, maxsplit=-1)`  
- No argument → splits on **any whitespace**, strips leading/trailing whitespace, no empty strings  
- `sep=','` → splits on comma exactly  
- `maxsplit` → limits number of splits

In [13]:
# Split on space (default)
log = "2024-01-15 ERROR user=john action=login status=failed"
print(log.split())            # ['2024-01-15', 'ERROR', 'user=john', ...]

# split() handles multiple spaces
print("  hello   world  ".split())    # ['hello', 'world']

# maxsplit — stop after N splits
print(log.split(",", maxsplit=1))    # ['2024-01-15', 'ERROR,...']

['2024-01-15', 'ERROR', 'user=john', 'action=login', 'status=failed']
['hello', 'world']
['2024-01-15 ERROR user=john action=login status=failed']


In [14]:
# Split log line into parts
log = "2024-01-15 ERROR user=john action=login status=failed"
parts = log.split()
print(parts)

# Unpack fixed-position fields, rest goes into a variable
date, level, *rest = parts
print(f"date   = {date}")
print(f"level  = {level}")
print(f"rest   = {' '.join(rest)}")

['2024-01-15', 'ERROR', 'user=john', 'action=login', 'status=failed']
date   = 2024-01-15
level  = ERROR
rest   = user=john action=login status=failed


In [15]:
# split on comma — spaces NOT removed automatically
csv = "2024-01-15,ERROR, user=john, action=login"
raw  = csv.split(",")
clean = [p.strip() for p in csv.split(",")]    # strip each part

print(f"split on ','  → {raw}")
print(f"split + strip → {clean}")

split on ','  → ['2024-01-15', 'ERROR', ' user=john', ' action=login']
split + strip → ['2024-01-15', 'ERROR', 'user=john', 'action=login']


---
## 5. join() — Combine a List Into a String

`separator.join(iterable)` — the separator is called **on the separator string**, not on the list.  
All elements must be strings.

In [17]:
parts = ["2024-01-15", "ERROR", "user=john"]
print(",".join(parts))     # 2024-01-15,ERROR,user=john

cols = ["Alice", "Engineering", "95000"]
print("|".join(cols))      # Alice|Engineering|95000
print(" ".join(cols))      # Alice Engineering 95000
print("-".join(cols))      # Alice-Engineering-95000

2024-01-15,ERROR,user=john
Alice|Engineering|95000
Alice Engineering 95000
Alice-Engineering-95000


In [19]:
# split → transform → join pattern — very common in DE
row = "  Alice , Engineering , 95000  "

# Clean each field and rebuild as proper CSV
cleaned = ",".join(p.strip() for p in row.split(","))
print(cleaned)   # Alice,Engineering,95000

# Equivalent without generator:
parts   = [p.strip() for p in row.split(",")]
cleaned = ",".join(parts)
print(cleaned)

Alice,Engineering,95000
Alice,Engineering,95000


---
## 6. find() / index() / count() / replace()

| Method | Returns | On miss |
|--------|---------|--------|
| `find(sub)` | First index | `-1` |
| `index(sub)` | First index | `ValueError` |
| `count(sub)` | Number of non-overlapping occurrences | `0` |
| `replace(old, new)` | New string with all replacements | original if not found |

In [21]:
text = "user=john action=login status=failed action=retry"

print(f"find('action')  → {text.find('action')}")    # 11
print(f"find('xyz')     → {text.find('xyz')}")        # -1
print(f"count('action') → {text.count('action')}")    # 2

find('action')  → 10
find('xyz')     → -1
count('action') → 2


In [22]:
text = "user=john action=login status=failed action=retry"

# replace all
print(text.replace("action=", "event="))        # replaces both

# replace only the first occurrence
print(text.replace("action=", "event=", 1))     # replaces only first

user=john event=login status=failed event=retry
user=john event=login status=failed action=retry


In [24]:
# DE pattern: chain replace to remove multiple characters
phones = [
    "+91-98765-43210",
    "(987) 654-3210",
    "987.654.3210",
]

for phone in phones:
    clean = phone.replace("+91", "").replace("-", "").replace(" ", "").replace("(", "").replace(")", "").replace(".", "")
    print(clean)

9876543210
9876543210
9876543210


---
## 7. startswith / endswith / in — Membership Checks

In [25]:
filename = "orders_2024_01.csv"

print(filename.startswith("orders"))              # True
print(filename.endswith(".csv"))                  # True
print(filename.endswith((".csv", ".tsv")))        # True — tuple of allowed suffixes
print(".csv" in filename)                         # True — substring check
print(".parquet" in filename)                     # False

True
True
True
True
False


In [26]:
# DE use case: filter file names by extension
files = [
    "orders_2024_01.csv",
    "customers_2024.csv",
    "products.parquet",
    "config.json",
    "schema.sql",
]

csv_files = [f for f in files if f.endswith(".csv")]
print(f"CSV files:     {csv_files}")

# Filter log lines containing errors
logs = [
    "2024-01-15 INFO pipeline started",
    "2024-01-15 ERROR disk full",
    "2024-01-15 INFO batch complete",
]

errors = [line for line in logs if "ERROR" in line]
print(f"ERROR lines:   {errors}")

CSV files:     ['orders_2024_01.csv', 'customers_2024.csv']
ERROR lines:   ['2024-01-15 ERROR disk full']


---
## 8. String Slicing — Extract by Position

`s[start:stop:step]` — start is inclusive, stop is exclusive, both are 0-based.

In [28]:
s = "2024-01-15 ERROR user=john"

print(f"year  = {s[0:4]}")    # 2024
print(f"month = {s[5:7]}")    # 01
print(f"day   = {s[8:10]}")   # 15
print(f"date  = {s[:10]}")    # 2024-01-15
print(f"rest  = {s[11:]}")    # ERROR user=john
print(f"last4 = {s[-4:]}")    # john
print(f"rev   = {s[::-1]}")   # reversed

year  = 2024
month = 01
day   = 15
date  = 2024-01-15
rest  = ERROR user=john
last4 = john
rev   = nhoj=resu RORRE 51-10-4202


In [29]:
# Slice date strings in a list
dates = ["2024-01-15", "2024-02-20", "2024-03-05"]

year_months = [d[:7] for d in dates]   # '2024-01'
years       = [d[:4] for d in dates]   # '2024'

print(f"dates: {year_months}")
print(f"years: {years}")

dates: ['2024-01', '2024-02', '2024-03']
years: ['2024', '2024', '2024']


---
## 9. f-strings and String Formatting

In [30]:
name   = "Alice"
salary = 95000

print(f"{name} earns ${salary:,}")           # Alice earns $95,000
print(f"{salary:.2f}")                       # 95000.00
print(f"{name:<10} | {salary:>8,}")          # left-align name, right-align salary
print(f"{'HEADER':^20}")                     # centred in 20 chars

Alice earns $95,000
95000.00
Alice      |   95,000
       HEADER       


In [31]:
# Build a formatted report table
records = [
    {"id": 1, "name": "Alice Johnson",  "dept": "Engineering", "salary": 95000},
    {"id": 2, "name": "Bob Smith",      "dept": "Marketing",   "salary": 72000},
    {"id": 3, "name": "Carol White",    "dept": "Finance",     "salary": 85000},
]

print(f"{'ID':<4} {'Name':<20} {'Dept':<14} {'Salary':<10}")
print(f"{'---':<4} {'-------------------':<20} {'-------------':<14} {'----------':<10}")
for r in records:
    print(f"{r['id']:<4} {r['name']:<20} {r['dept']:<14} ${r['salary']:,<9}")

ID   Name                 Dept           Salary    
---  -------------------  -------------  ----------
1    Alice Johnson        Engineering    $95000,,,,
2    Bob Smith            Marketing      $72000,,,,
3    Carol White          Finance        $85000,,,,


---
## 10. isdigit / isalpha / isalnum — Character Type Checks

In [32]:
print(f"'12345' isdigit: {'12345'.isdigit()}")      # True
print(f"'12.5' isdigit: {'12.5'.isdigit()}")        # False — dot is not a digit
print(f"'Alice' isalpha: {'Alice'.isalpha()}")      # True
print(f"'Alice123' isalnum: {'Alice123'.isalnum()}")  # True
print(f"'Alice 123' isalnum: {'Alice 123'.isalnum()}")  # False — space

'12345' isdigit: True
'12.5' isdigit: False
'Alice' isalpha: True
'Alice123' isalnum: True
'Alice 123' isalnum: False


In [33]:
# Extract only digits from a phone number
phones = ["+91-98765-43210", "9876100002", "(098) 761-00003"]

for phone in phones:
    digits = "".join(c for c in phone if c.isdigit())
    print(f"digits only: {digits}")

digits only: 919876543210
digits only: 9876100002
digits only: 09876100003


---
## 11. Combined Patterns — Real DE Usage

In [ ]:
# Pattern 1: Parse key=value log line into a dict
log = "2024-01-15 ERROR user=john action=login status=failed"

parts  = log.split()
result = {"date": parts[0], "level": parts[1]}

for token in parts[2:]:
    if "=" in token:
        key, value = token.split("=", 1)    # maxsplit=1 handles value containing '='
        result[key] = value

print(result)

{'date': '2024-01-15', 'level': 'ERROR', 'user': 'john', 'action': 'login', 'status': 'failed'}


In [ ]:
# Pattern 2: Parse CSV row into dict (header + data)
def parse_csv_row(header_line: str, data_line: str) -> dict:
    headers = [h.strip() for h in header_line.split(",")]
    values  = [v.strip() for v in data_line.split(",")]
    return dict(zip(headers, values))

header = "customer_id, full_name, email, city"
data   = "101, Alice Johnson , alice@gmail.com , Mumbai"

record = parse_csv_row(header, data)
print(record)

{'customer_id': '101', 'full_name': 'Alice Johnson', 'email': 'alice@gmail.com', 'city': 'Mumbai'}


In [ ]:
# Pattern 3: Count email domains
emails = [
    "alice@gmail.com",
    "bob@yahoo.com",
    "carol@gmail.com",
    "dave@hotmail.com",
    "eve@gmail.com",
    "frank@yahoo.com",
]

domain_counts = {}
for email in emails:
    domain = email.strip().lower().split("@")[1]
    domain_counts[domain] = domain_counts.get(domain, 0) + 1

print("Domain counts (sorted by frequency):")
for domain, cnt in sorted(domain_counts.items(), key=lambda x: -x[1]):
    print(f"  {domain:<12} → {cnt}")

Domain counts (sorted by frequency):
  gmail.com   → 3
  yahoo.com   → 2
  hotmail.com → 1


---
## 12. Day 2 Problems — Official Solutions

Try writing your solution before running each cell.

In [ ]:
# Problem 1 (Easy) — Parse CSV row into dict
# Given a CSV row as a string, parse it into a dictionary of field-name → value using split()

def parse_csv_row(row: str, headers: list) -> dict:
    values = [v.strip() for v in row.split(",")]
    return dict(zip(headers, values))

headers = ["customer_id", "name", "email", "city"]
rows = [
    "101, Alice Johnson, alice@gmail.com, Mumbai",
    "102, Bob Smith, bob@yahoo.com, Delhi",
]

for row in rows:
    print(parse_csv_row(row, headers))

{'customer_id': '101', 'name': 'Alice Johnson', 'email': 'alice@gmail.com', 'city': 'Mumbai'}
{'customer_id': '102', 'name': 'Bob Smith', 'email': 'bob@yahoo.com', 'city': 'Delhi'}


In [ ]:
# Problem 2 (Easy) — Email domain counter
emails = [
    "alice@gmail.com", "bob@yahoo.com", "carol@gmail.com",
    "dave@hotmail.com", "eve@gmail.com", "frank@yahoo.com",
]

domain_counts = {}
for email in emails:
    domain = email.strip().lower().split("@")[1]
    domain_counts[domain] = domain_counts.get(domain, 0) + 1

for domain, cnt in sorted(domain_counts.items(), key=lambda x: -x[1]):
    print(f"{domain:<12}: {cnt}")

gmail.com   : 3
yahoo.com   : 2
hotmail.com : 1


In [ ]:
# Problem 3 (Medium) — Parse structured log line
log = "2024-01-15 ERROR user=john action=login status=failed"

parts  = log.split()
result = {"date": parts[0], "level": parts[1]}

for token in parts[2:]:
    if "=" in token:
        key, value = token.split("=", 1)
        result[key] = value

print(result)

{'date': '2024-01-15', 'level': 'ERROR', 'user': 'john', 'action': 'login', 'status': 'failed'}


---
## 13. Interview Gotchas — Know These Cold

In [ ]:
# Gotcha 1: split() vs split(' ') — very different behaviour
s = "  hello   world  "

print(f"split()    → {s.split()}")      # ['hello', 'world']  ← handles multiple spaces
print(f"split(' ') → {s.split(' ')}")   # lots of empty strings ← fragile

split()    → ['hello', 'world']  (2 parts — strips leading/trailing whitespace)
split(' ') → ['', '', 'hello', '', '', 'world', '', '']  (8 parts — empty strings!)


In [ ]:
# Gotcha 2: find() returns -1 on miss (not None, not False)
s = "hello world"
print(f"find returns: {s.find('xyz')}")    # -1

# Gotcha 3: strip() only removes edges, not middle spaces
s = "hello world"
print(f"strip only removes edges: {repr(s.strip())}")           # 'hello world'
print(f"replace removes all: {repr(s.replace(' ', ''))}")       # 'helloworld'

find returns: -1
strip only removes edges: 'hello world'
replace removes all: 'helloworld'


In [ ]:
# Gotcha 4: isdigit() returns False for floats and negative numbers
print("123".isdigit())    # True
print("-123".isdigit())   # False — minus sign fails
# For numeric check: use try/except float()
def is_numeric(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

True
False


In [ ]:
# Gotcha 5: split name — what if middle name exists?
name = "Alice Johnson"
print(name)    # safe

# Safe split: maxsplit=1 ensures last name gets everything after first space
first, last = name.split(maxsplit=1)
print(f"first={first}  last={last}")

Alice Johnson
first=Alice  last=Johnson


---
## Quick Cheat Sheet

```python
# Case
s.upper()  s.lower()  s.title()  s.capitalize()

# Strip
s.strip()  s.lstrip()  s.rstrip()  s.strip("#")

# Split / Join
s.split(",")           # split on comma
s.split(maxsplit=1)    # max 1 split
s.split()              # whitespace, strips
",".join(lst)          # join list → string

# Find / Replace / Count
s.find("x")            # first index, -1 if miss
s.count("x")           # occurrences
s.replace("o", "0")    # all replacements
s.replace("o", "0", 1) # first only

# Check
s.startswith("pre")    s.endswith(".csv")
"sub" in s             s.isdigit()  s.isalpha()

# Slice
s[0:4]   s[-1]   s[::-1]   s[5:]

# Format
f"{val:,}"   f"{val:.2f}"   f"{val:<10}"

# Useful one-liners
"".join(c for c in s if c.isdigit())   # digits only
[p.strip() for p in s.split(",")]      # split + strip each part
email.split("@")[1]                    # domain from email
s.split(maxsplit=1)                    # first word + rest
```